# 체형·표면·옷 처리 완전 요인 실험 (16 조합) + 최선 조합 250 프레임 확인
| 요인 | 수준 |
|---|---|
| β (체형) | `b0` 기존 β / `b1` 개인 β |
| D (표면) | `d0` 없음 / `ds` 신발만 / `du` 상의·머리만 / `da` 신발+상의·머리 (반바지 D 는 모든 조합에서 0) |
| 데이터 항 | `t2` 양방향 / `t1` 하의 한쪽 (BUFF·DoubleFusion 방식) |

- 평가: 002 세션 앞 84 프레임, 다리 × 구간(전체·디딤기·흔들기) RMSE / 편향 / 상관 + 요인별 주효과
- **선택 기준 (미리 고정): 두 다리 RMSE 평균 최소, 같으면 최대 오차 최소** → 그 조합으로 250 프레임 실행
- 끊기면 다시 실행: 끝난 조합은 건너뜀. 런타임: GPU

In [ ]:
#@title 설정
ROOT       = "/content/drive/MyDrive/walking_0729"   #@param {type:"string"}
SESSION    = "0729_walk_3kmh_5500"                   #@param {type:"string"}
IMU_PREFIX = "MT_01200295_002_60fps_walking"         #@param {type:"string"}
QUICK      = 84     #@param {type:"integer"}
RUN_FULL_BEST = True  #@param {type:"boolean"}

from google.colab import drive
drive.mount("/content/drive")
import os, sys, json, glob, time, shutil, subprocess, itertools
import numpy as np
SESS = f"{ROOT}/sessions/{SESSION}"; S = f"{ROOT}/scripts"
EXP = f"{SESS}/experiments"; os.makedirs(f"{EXP}/logs", exist_ok=True)
VARIANTS = [(b, d, t) for t in ("t1", "t2") for b in ("b0", "b1") for d in ("d0", "ds", "du", "da")]
print(len(VARIANTS), "조합:", " ".join("_".join(v) for v in VARIANTS))
import torch; assert torch.cuda.is_available(), "GPU 런타임 필요"
!pip -q install smplx imageio-ffmpeg


In [ ]:
#@title 1. depth/video 를 Colab 로컬 디스크로 복사 (Drive 읽기 병목 제거, 끊기면 재연결 후 재시도)
def _copy(src, g):
    if os.path.exists(g) and os.path.getsize(g) == os.path.getsize(src):
        return
    for attempt in range(3):
        try:
            shutil.copy(src, g + ".part"); os.replace(g + ".part", g); return
        except OSError as e:
            print(f"  복사 실패 {os.path.basename(src)} ({e}) → Drive 재연결 {attempt + 1}/3")
            try: drive.flush_and_unmount()
            except Exception: pass
            drive.mount("/content/drive", force_remount=True); time.sleep(5)
    raise RuntimeError(src)

CACHE = f"/content/walk_cache/{SESSION}"
def make_cache(n):
    os.makedirs(f"{CACHE}/out", exist_ok=True); os.makedirs(f"{CACHE}/video", exist_ok=True)
    dfiles = sorted(glob.glob(f"{SESS}/01_depth/work/out/*_person_metric.npy"))
    if n > 0: dfiles = dfiles[:n]
    for i, f in enumerate(dfiles):
        _copy(f, f"{CACHE}/out/{os.path.basename(f)}")
        v = f"{SESS}/01_depth/work/video/{os.path.basename(f).replace('_person_metric.npy', '')}.npz"
        if os.path.exists(v): _copy(v, f"{CACHE}/video/{os.path.basename(v)}")
    print(f"[cache] {len(dfiles)} 프레임 준비")
make_cache(QUICK)


In [ ]:
#@title 2. 실행기 (조합 하나 = A0 → near → far → 튐 재피팅 → NB6 → NB7)
def tag_of(name, onesided, frames):
    return f"_exp_{name}" + ("_c1" if onesided else "") + (f"_q{frames}" if frames > 0 else "")

def run_variant(b, d, t, frames):
    name = f"{b}_{d}_{t}"; onesided = (t == "t1"); tag = tag_of(name, onesided, frames)
    done = f"{SESS}/06_imu{tag}/imu_validation.json"
    if os.path.exists(done):
        print(f"[skip] {name} ({frames or 'full'}) 이미 완료"); return tag
    env = os.environ.copy()
    env.update(WALK_ROOT=ROOT, WALK_SESSION=SESSION, WALK_DEPTH_SEQ=f"{CACHE}/out", WALK_VIDEO_SEQ=f"{CACHE}/video",
               WALK_PERSONAL=f"{ROOT}/assets/exp/{b}_{d}.npz", WALK_FIT_TAG=f"_exp_{name}",
               WALK_CLOTH_ONESIDED="1" if onesided else "0", WALK_FRAME_LIMIT=str(frames))
    log = open(f"{EXP}/logs/{name}{'_q'+str(frames) if frames else '_full'}.log", "w")
    def sh(cmd):
        p = subprocess.run(cmd, shell=True, cwd=S, env=env, stdout=log, stderr=subprocess.STDOUT)
        if p.returncode != 0:
            raise RuntimeError(f"{name}: '{cmd}' 실패 → {log.name}")
    t0 = time.time(); print(f"[run] {name} ({frames or 'full'} 프레임) ...", flush=True)
    sh("python a0_align_v2.py")
    sh("python depthfit_near_leg_lam0.py --stage f3")
    sh("python depthfit_far_leg_lam0.py --stage far2")
    sh("python far_spike_refit.py")
    n = np.load(f"{SESS}/04_fit{tag}/near/depthfit_near_leg_lam0_f3.npz"); k = n["knee_fit"] - n["knee_fit"].mean()
    ac = np.correlate(k, k, "full")[len(k) - 1:]; ac /= ac[0]
    m = json.load(open(f"{SESS}/00_extract/meta.json")); fps = (m.get("window") or {}).get("fps_eff") or m["fps"]
    lo, hi = int(0.7 * fps), min(len(ac) - 1, int(2.0 * fps)); env["WALK_GAIT_FRAMES"] = str(lo + int(np.argmax(ac[lo:hi])))
    for s_ in ("knee_measurement_basis.py", "far_thigh_npts_gate.py", "far_thigh_infill.py", "far_thigh_infill_v2.py"):
        sh(f"python {s_}")
    sh(f"python imu_validate.py --imu-prefix {IMU_PREFIX}")
    log.close(); print(f"      완료 {(time.time() - t0) / 60:.1f}분", flush=True)
    return tag

def metrics(tag):
    v = np.load(f"{SESS}/06_imu{tag}/imu_validation.npz", allow_pickle=True)
    out = {}
    for s_, leg in (("R", "오른"), ("L", "왼")):
        a, i = v["smpl_" + s_].astype(float), v["imu_" + s_].astype(float); e = a - i; ok = np.isfinite(e)
        for ph, mk in (("전체", ok), ("디딤기", ok & (i < 20)), ("흔들기", ok & (i >= 20))):
            out[f"{leg}_{ph}_RMSE"] = float(np.sqrt(np.mean(e[mk] ** 2))); out[f"{leg}_{ph}_bias"] = float(np.mean(e[mk]))
        out[f"{leg}_r"] = float(np.corrcoef(a[ok], i[ok])[0, 1]); out[f"{leg}_max"] = float(np.max(np.abs(e[ok])))
    nr = np.load(f"{SESS}/04_fit{tag}/near/depthfit_near_leg_lam0_f3.npz")
    out["오른_표면잔차cm"] = float(np.median(nr["resid_after_cm"]))
    far = np.load(f"{SESS}/04_fit{tag}/far/depthfit_far_leg_lam0_far2.npz", allow_pickle=True)
    out["재피팅"] = int(far["refit"].sum()) if "refit" in far.files else 0; out["reject"] = int(far["reject"].sum()) if "reject" in far.files else 0
    out["offset_s"] = float(v["offset_s"]); out["평균RMSE"] = (out["오른_전체_RMSE"] + out["왼_전체_RMSE"]) / 2
    return out


In [ ]:
#@title 3. 16 조합 실행 (약 15~20분 × 16, 끊기면 다시 실행 — 끝난 조합은 건너뜀)
T_ALL = time.time()
for i, (b, d, t) in enumerate(VARIANTS):
    print(f"--- {i + 1}/{len(VARIANTS)}", end=" ")
    run_variant(b, d, t, QUICK)
print(f"✅ 16 조합 완료 ({(time.time() - T_ALL) / 60:.0f}분)")


In [ ]:
#@title 4. 요약표 + 요인별 주효과 + 최선 조합 선택
import pandas as pd
rows = []
for b, d, t in VARIANTS:
    tag = tag_of(f"{b}_{d}_{t}", t == "t1", QUICK)
    if os.path.exists(f"{SESS}/06_imu{tag}/imu_validation.npz"):
        rows.append(dict(조합=f"{b}_{d}_{t}", β=b, D=d, 항=t, **metrics(tag)))
df = pd.DataFrame(rows)
show = ["조합", "평균RMSE", "오른_전체_RMSE", "오른_디딤기_RMSE", "오른_흔들기_RMSE", "왼_전체_RMSE", "왼_디딤기_RMSE", "왼_흔들기_RMSE",
        "오른_전체_bias", "왼_전체_bias", "오른_r", "왼_r", "오른_max", "왼_max", "오른_표면잔차cm", "재피팅", "reject", "offset_s"]
pd.set_option("display.width", 250); pd.set_option("display.precision", 2)
print(df[show].sort_values("평균RMSE").to_string(index=False))
eff = []
for f_, lv in (("β", ("b0", "b1")), ("D", ("d0", "ds", "du", "da")), ("항", ("t2", "t1"))):
    for l in lv:
        sub = df[df[f_] == l]
        eff.append(dict(요인=f_, 수준=l, n=len(sub), 오른=sub["오른_전체_RMSE"].mean(), 왼=sub["왼_전체_RMSE"].mean(),
                        오른디딤=sub["오른_디딤기_RMSE"].mean(), 왼흔들=sub["왼_흔들기_RMSE"].mean(), 평균=sub["평균RMSE"].mean()))
eff = pd.DataFrame(eff); print("\n[주효과] 각 수준의 평균 RMSE (다른 요인은 모든 수준 평균)"); print(eff.to_string(index=False))
best = df.sort_values(["평균RMSE", "오른_max"]).iloc[0]
print(f"\n★ 선택 (기준: 두 다리 평균 RMSE 최소, 동률이면 최대 오차 최소): {best['조합']}  평균 {best['평균RMSE']:.2f}° (오른 {best['오른_전체_RMSE']:.2f} / 왼 {best['왼_전체_RMSE']:.2f})")
df.to_csv(f"{EXP}/summary_q{QUICK}.csv", index=False, encoding="utf-8-sig"); eff.to_csv(f"{EXP}/main_effects_q{QUICK}.csv", index=False, encoding="utf-8-sig")
json.dump(dict(best=best["조합"], criterion="mean RMSE (R,L) min, tie → R max error min", quick=QUICK), open(f"{EXP}/best.json", "w"))
print("저장:", EXP)


In [ ]:
#@title 5. 최선 조합으로 250 프레임 확인 (약 45분) + 기존 β 250 프레임과 비교
if RUN_FULL_BEST:
    b, d, t = json.load(open(f"{EXP}/best.json"))["best"].split("_")
    make_cache(0)
    tag = run_variant(b, d, t, 0)
    full = metrics(tag)
    base = np.load(f"{SESS}/06_imu/imu_validation.npz", allow_pickle=True)
    def bm(s_, mk=None):
        a, i = base["smpl_" + s_].astype(float), base["imu_" + s_].astype(float); e = a - i; ok = np.isfinite(e) if mk is None else (np.isfinite(e) & mk(i))
        return float(np.sqrt(np.mean(e[ok] ** 2)))
    print(f"250 프레임 비교  (기존 β → {b}_{d}_{t})")
    for s_, leg in (("R", "오른"), ("L", "왼")):
        print(f"  {leg}: 전체 {bm(s_):.2f} → {full[leg + '_전체_RMSE']:.2f}°  | 디딤기 {bm(s_, lambda i: i < 20):.2f} → {full[leg + '_디딤기_RMSE']:.2f}°"
              f"  | 흔들기 {bm(s_, lambda i: i >= 20):.2f} → {full[leg + '_흔들기_RMSE']:.2f}°  | r {full[leg + '_r']:.3f}")
    json.dump(full, open(f"{EXP}/best_full250.json", "w"), ensure_ascii=False, indent=1)
